# Agente LangGraph + MCP de Playwright: perfil de LinkedIn

Parámetro: el **nombre de usuario** (`linkedin.com/in/<usuario>`).

1. **Login:** el agente escribe las credenciales del `.env` (el LLM no las ve) y pulsa Iniciar sesión. Si aparece un captcha o código, **lo resuelves tú** en la ventana de Chrome (hasta 5 minutos).
2. **Extracción:** el agente abre el perfil → **Más** → **Guardar en PDF**; el PDF se lee en una carpeta temporal, se convierte en JSON sin LLM y **se borra**.

- Solo perdura la sesión de Chrome (`.cache/linkedin_mcp_profile`); no hace falta repetir el login mientras siga vigente.
- Si interrumpes una celda a medias, reinicia el kernel.
- El JSON mostrado contiene datos personales: limpia las salidas antes de hacer commit.

In [ ]:
import os, sys, json
if not os.path.isdir("app"):
    os.chdir(os.path.abspath(".."))  # raíz del repo
sys.path.insert(0, os.getcwd())

from app.infrastructure.outbound.extractors.linkedin.agent_extractor import LinkedinAgentExtractor

USERNAME = "rosanaferrero"

extractor = LinkedinAgentExtractor(verbose=True)  # verbose: imprime cada acción del agente

In [ ]:
# 1) LOGIN: si aparece el captcha, resuélvelo en la ventana de Chrome
login = await extractor.iniciar_sesion()
print("\nLOGIN:", login.status.value, "|", login.details)

In [ ]:
# 2) EXTRACCIÓN: descarga el PDF, lo pasa a JSON y lo borra
resultado = await extractor.extract(USERNAME)

audit = resultado["audit"]
print("\nESTADO:", audit.status.value, "|", audit.duration_ms, "ms")
if audit.details:
    print("DETALLE:", audit.details)

print(json.dumps(resultado["data"], ensure_ascii=False, indent=2))